# PatchDistill Colab Pilot

Run the clone/setup cells first. They copy the whole PatchDistill repository into the Colab runtime, then install dependencies from the repository root.

If `!pwd` is `/content` and `!ls` only shows `sample_data`, the repository is not on the Colab runtime yet. Opening this notebook from Cursor does not automatically copy `/home/blackleg/ws/research/llm/new/patchDistill` to Colab.

The next code cell uses the GitHub route:

```bash
git clone https://github.com/black-leg-nameko/patchDistill.git /content/patchDistill
cd /content/patchDistill
```

```python
# Route B: put the folder on Google Drive, then mount Drive
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/patchDistill
```

After that, the install cell below will find `requirements.txt` and `patchdistill/`.

In [ ]:
!nvidia-smi

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/black-leg-nameko/patchDistill.git"
REPO_DIR = Path("/content/patchDistill")
FORCE_SYNC_REPO = True  # Reset Colab checkout to origin/main before each run.

if REPO_DIR.exists():
    print(f"Repository already exists at {REPO_DIR}; fetching latest origin/main.")
    subprocess.check_call(["git", "-C", str(REPO_DIR), "fetch", "origin", "main"])
    if FORCE_SYNC_REPO:
        print("Resetting checkout to origin/main. Ignored runs/ files are preserved until the cleanup cell.")
        subprocess.check_call(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/main"])
    else:
        subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])
else:
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])

os.chdir(REPO_DIR)
print("Using repo root:", Path.cwd())
print("Current commit:")
subprocess.check_call(["git", "log", "--oneline", "-1"])
print("Top-level files:")
print("\n".join(sorted(p.name for p in Path.cwd().iterdir())[:40]))

In [ ]:
MODEL_NAME = "gpt2"
RUN_NAME = "gpt2_a100_contrastive_frame_001"
LAYERS = "0,6,11"
DATA_PROFILE = "contrastive_frame"
SEED = 31
N_DATA = 400
MAX_FEATURE_EXAMPLES = 240
MAX_PATCH_EXAMPLES = 32
MAX_PATCH_POSITIONS = 3
DTYPE = "bfloat16"
ATTN_IMPL = None
SURROGATE_SPLIT = "group"
DETECTOR_SPLIT = "group"
DATA_PATH = f"data/synthetic_direct_pi_{DATA_PROFILE}.jsonl"
SURROGATE_RUN_DIR = f"runs/surrogate_{DATA_PROFILE}"
CLEAN_RUNS = True

# For a stronger A100 pilot, try the 0.5B Qwen model:
# MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
# RUN_NAME = "qwen25_05b_a100_contrastive_frame_001"
# LAYERS = "0,12,23"
# ATTN_IMPL = "eager"

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

def find_repo_root():
    candidates = [Path.cwd(), Path('/content/patchDistill')]
    drive = Path('/content/drive/MyDrive')
    if drive.exists():
        candidates.extend([
            drive / 'patchDistill',
            drive / 'Colab Notebooks' / 'patchDistill',
        ])
    for candidate in candidates:
        if (candidate / 'requirements.txt').exists() and (candidate / 'patchdistill').is_dir():
            return candidate
    for root in [Path('/content'), drive]:
        if root.exists():
            for req in root.rglob('requirements.txt'):
                candidate = req.parent
                if (candidate / 'patchdistill').is_dir():
                    return candidate
    return None

repo_root = find_repo_root()
if repo_root is None:
    raise FileNotFoundError(
        'PatchDistill repo root was not found. Current Colab runtime does not contain the project files. '
        'If !pwd is /content and !ls only shows sample_data, clone/upload the whole repository first. '
        'Expected files: requirements.txt and patchdistill/.'
    )

os.chdir(repo_root)
print('Using repo root:', Path.cwd())
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'])

In [ ]:
from pathlib import Path
import shutil

if CLEAN_RUNS:
    shutil.rmtree("runs", ignore_errors=True)
    Path("runs").mkdir(parents=True, exist_ok=True)
    print("Cleaned runs/ for an isolated experiment archive.")
else:
    print("Keeping existing runs/ contents.")

In [ ]:
!python -m patchdistill.cli make-data --n {N_DATA} --profile {DATA_PROFILE} --seed {SEED} --out {DATA_PATH}
!python -m patchdistill.cli run-surrogate --data {DATA_PATH} --out {SURROGATE_RUN_DIR} --split {SURROGATE_SPLIT} --seed {SEED}

In [ ]:
attn_arg = "" if ATTN_IMPL is None else f"--attn-implementation {ATTN_IMPL}"
!python -m patchdistill.cli hf-extract \
  --model {MODEL_NAME} \
  --data {DATA_PATH} \
  --out runs/{RUN_NAME}_features.jsonl \
  --max-examples {MAX_FEATURE_EXAMPLES} \
  --layers {LAYERS} \
  --dtype {DTYPE} \
  {attn_arg}

In [ ]:
!python -m patchdistill.cli hf-patch \
  --model {MODEL_NAME} \
  --data {DATA_PATH} \
  --out runs/{RUN_NAME}_patch.jsonl \
  --layers {LAYERS} \
  --max-examples {MAX_PATCH_EXAMPLES} \
  --max-positions {MAX_PATCH_POSITIONS} \
  --dtype {DTYPE} \
  {attn_arg}

In [ ]:
!python -m patchdistill.cli fit-proxy \
  --features runs/{RUN_NAME}_features.jsonl \
  --patch runs/{RUN_NAME}_patch.jsonl \
  --out runs/{RUN_NAME}_proxy

!python -m patchdistill.cli fit-detector \
  --features runs/{RUN_NAME}_features.jsonl \
  --out runs/{RUN_NAME}_detector_features_only \
  --split {DETECTOR_SPLIT}

!python -m patchdistill.cli fit-detector \
  --features runs/{RUN_NAME}_features.jsonl \
  --patch runs/{RUN_NAME}_patch.jsonl \
  --out runs/{RUN_NAME}_detector_distilled \
  --split {DETECTOR_SPLIT}

In [ ]:
!python -m patchdistill.cli collect-results --runs runs --out runs/summary.json --markdown runs/summary.md

from pathlib import Path
print(Path("runs/summary.md").read_text())

## Manual Result Review

This notebook no longer pushes results to GitHub automatically. Save the notebook after running, then inspect the printed summary from Cursor/Codex and manually update the repository notes/paper.


In [ ]:
from pathlib import Path
import tarfile

!python -m patchdistill.cli collect-results --runs runs --out runs/summary.json --markdown runs/summary.md
summary = Path("runs/summary.md").read_text()
print(summary)

bundle_path = Path("runs/patchdistill_results_bundle.tar.gz")
with tarfile.open(bundle_path, "w:gz") as tar:
    for path in sorted(Path("runs").rglob("*")):
        if path.is_file() and path != bundle_path:
            tar.add(path, arcname=path)

print("\nRESULT_BUNDLE:", bundle_path.resolve())
print("RESULT_FILES:")
for path in sorted(Path("runs").rglob("*")):
    if path.is_file():
        print(path)

# In Colab, uncomment these lines if you want to download the raw result archive.
# from google.colab import files
# files.download(str(bundle_path))
